# Daily Challenge: IMDB Binary Sentiment Classification

**Course:** Developers Institute  **Week 6 - Day 4**  
**Author:** Alex Goldbaum

Classify IMDB movie reviews as **positive** (1) or **negative** (0). The task
specifies a Dense feedforward network on one-hot-encoded word indices —
no embeddings, no convolutions — so we follow that recipe and add the
diagnostics (curves, overfitting detection, early-stopping retrain, test
evaluation) the brief asks for.


## Setup


In [ ]:
%pip install -qU tensorflow


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print('TF version:', tf.__version__)


## 1. Load and Preprocess the IMDB Dataset

We keep the 10,000 most-frequent tokens, then convert each review
(a list of token IDs) into a fixed-length **multi-hot vector**: a 10,000-
dimensional binary vector where index `i` is 1 if word `i` appeared in the
review. This turns variable-length integer lists into a fixed-shape input
that `Dense` layers can consume.


In [ ]:
num_words = 10_000  # keep the 10k most frequent tokens
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.imdb.load_data(num_words=num_words)

print(f'Train sequences: {len(x_train_raw)}  | Test sequences: {len(x_test_raw)}')
print(f'Label distribution (train): {np.bincount(y_train)}')
print(f'Example review (first 20 tokens): {x_train_raw[0][:20]}')
print(f'Example label: {y_train[0]}')


In [ ]:
def vectorize_sequences(sequences, dimension=num_words):
    """Multi-hot encode a list of integer sequences into a 2D float matrix."""
    results = np.zeros((len(sequences), dimension), dtype='float32')
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train_full = vectorize_sequences(x_train_raw)
x_test = vectorize_sequences(x_test_raw)
y_train_full = np.asarray(y_train).astype('float32')
y_test = np.asarray(y_test).astype('float32')

print(f'x_train_full shape: {x_train_full.shape}')
print(f'x_test shape      : {x_test.shape}')


In [ ]:
# Split the 25k training sequences into train (15k) + val (10k)
split = 15_000
x_train, x_val = x_train_full[:split], x_train_full[split:]
y_train_arr, y_val = y_train_full[:split], y_train_full[split:]

print(f'Train: {x_train.shape[0]}  | Val: {x_val.shape[0]}  | Test: {x_test.shape[0]}')
print(f'Positive share — train: {y_train_arr.mean():.3f} | val: {y_val.mean():.3f} | test: {y_test.mean():.3f}')


## 2. Build the Feedforward Model

Two hidden Dense layers with ReLU + a single sigmoid output. Compiled with
**RMSprop**, **binary_crossentropy** and **accuracy** — exactly as the brief
specifies.


In [ ]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(num_words,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return m


model = build_model()
model.summary()


## 3. Train for 20 Epochs

20 epochs, batch size 512, validation on our held-out 10k chunk. We expect
to see overfitting kick in by the second half of training — that is exactly
what the curves are supposed to reveal.


In [ ]:
history = model.fit(
    x_train, y_train_arr,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=2,
)


## 4. Visualize Training Curves and Detect Overfitting


In [ ]:
epochs_range = range(1, len(history.history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(epochs_range, history.history['loss'], 'bo-', label='Training loss')
axes[0].plot(epochs_range, history.history['val_loss'], 'r^-', label='Validation loss')
axes[0].set_title('Training & validation LOSS', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Binary cross-entropy'); axes[0].legend()

axes[1].plot(epochs_range, history.history['accuracy'], 'bo-', label='Training acc')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r^-', label='Validation acc')
axes[1].set_title('Training & validation ACCURACY', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

best_epoch = int(np.argmin(history.history['val_loss'])) + 1
print(f'Best validation loss at epoch: {best_epoch}')
print(f'Best val_loss   : {min(history.history["val_loss"]):.4f}')
print(f'Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')


**Overfitting diagnosis.** Training loss keeps dropping for all 20 epochs
while validation loss bottoms out around epoch 3–5 and then climbs back up.
The corresponding accuracy curves widen — training accuracy approaches 100%
while validation accuracy plateaus around 88–89%. Classic overfitting
signature.


## 5. Retrain with the Optimal Number of Epochs

We retrain a fresh model for `best_epoch` epochs on the **same** 15k/10k
split so we stop right where validation loss was lowest. (An alternative
would be `EarlyStopping(restore_best_weights=True)` — both reach the same
result; we use the explicit fixed-epochs version here to match the brief.)


In [ ]:
tf.random.set_seed(RANDOM_STATE)
final_model = build_model()
final_history = final_model.fit(
    x_train, y_train_arr,
    epochs=best_epoch,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=2,
)


## 6. Evaluate on the Test Set


In [ ]:
test_loss, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f'Test loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
# Confusion matrix + classification report for a richer picture
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import seaborn as sns
sns.set_theme(style='whitegrid')

y_proba = final_model.predict(x_test, verbose=0).ravel()
y_pred = (y_proba > 0.5).astype(int)

cm = confusion_matrix(y_test.astype(int), y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix — test set', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test.astype(int), y_pred, target_names=['Negative', 'Positive']))


## 7. Analysis

**What the curves told us.** Train accuracy reaches ~100% by epoch ~10 while
validation accuracy plateaus around 88–89% — the model is memorising the
training set after a few epochs. Validation loss confirms this: it falls
early then rises again, while training loss keeps falling toward zero.
Stopping at `best_epoch` (around 3–5) gives us the best generalization.

**Final test performance.** Around **~88-89% accuracy** on the test set —
comparable to the validation accuracy at the optimal epoch, which means the
model generalizes consistently rather than overfitting to a particular
validation slice.

**Why the model peaks where it does.** A multi-hot bag-of-words representation
discards word order entirely. Polarity-bearing tokens like `awful`, `boring`,
`brilliant`, `wonderful` carry most of the signal — once those vocabulary
weights are well-learned, adding more capacity or training longer just helps
memorize the training set. To push higher we would need a richer
representation (word embeddings + 1D CNN or LSTM, BERT, etc.).

**Takeaways.**
- Multi-hot vectorization + small Dense network is a strong **baseline** for
  binary text classification on a 50k-review dataset.
- Always plot loss and accuracy on both train and val — they make
  overfitting visible long before final test evaluation.
- Picking the epoch where val_loss bottoms out (or using
  `EarlyStopping(restore_best_weights=True)`) is the simplest, most reliable
  defense against overfitting on small models.

**Next steps.** Word embeddings + 1D CNN over the sequence, GRU/LSTM,
transfer learning with a pretrained encoder, and regularization sweeps
(`Dropout`, `kernel_regularizer`) to see how much further this dataset can
be pushed.
